# LitWBTrainingBasicConvnet notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/LitWBTrainingBasicConvnet/lit_wb_training_basic_convnet_notebook.ipynb)

This notebook trains **LitWBTrainingBasicConvnet**: what the project does and what it teaches is in its [README](README.md). Every project has its own notebook like this one, in its folder.

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), then 2 and 3 (4 and 5 to load or keep training a model) | Option A1 (`!python -m ...`) or Option B (`main(...)`) |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same command in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Run the project**](#2.-Run-the-project-(choose-one-option)) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed | `python -m LitWBTrainingBasicConvnet --config config01.json` from the repo root, with the `.venv` activated |
| **Colab CLI, for the final training run** | None, no notebook needed | `python -m LitWBTrainingBasicConvnet --config config01.json` on a Colab runtime opened from your terminal, with your env file loaded from Google Drive: see [**Training with the Colab CLI**](#Training-with-the-Colab-CLI) |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

The setup, paths, run-folder, and Lightning boilerplate that used to be copied into every project now comes from the course library [`colabutilscs4337`](https://txstcodeplayground.github.io/ColabUtilsCS4337/guide/course_projects.html): each setup cell below is one call into it, and the [README](README.md) lists what moved.

In Colab, run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU* (training also works on CPU, just slower).
2. **Install the course library** `colabutilscs4337`.
3. **Set up the repo:** one call mounts Google Drive (so your results survive runtime resets), clones (or updates) the repo, applies the Colab settings, installs only the missing requirements, and changes into the repo.
4. **Load your API keys** (optional) from Colab Secrets.
5. **Run LitWBTrainingBasicConvnet**, either as a terminal command or from Python, and look at the results.
6. **Load a trained model** later, without training again (section 4).
7. **Keep training** a run for more epochs, or finish one cut off by a disconnect (section 5).

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.

## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Run the project**](#2.-Run-the-project-(choose-one-option)).

Run these cells once per session (and again after the runtime resets): install the library, set up the repo (this mounts Drive, clones, installs, and changes directory), and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; it rewrites `.env`, so run the API-key cell again after it.

In [ ]:
# The course helpers (setup, env file, paths, Lightning plumbing) live in colabutilscs4337.
%pip install -q --upgrade colabutilscs4337

In [ ]:
import colabutilscs4337 as cu

# Mounts Drive, clones or updates the repo, copies .envcolab to .env, installs the
# requirements Colab doesn't have, and changes into the repo.
REPO = cu.project.setup_colab(
    "https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git"
)

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell calls `cu.env.load_secrets`: it looks for each key in the environment, in Colab Secrets (Colab may ask you to *Grant access*), in `.env`, and in `MyDrive/.colabutilscs4337/.env`. Keys it finds in Secrets are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.

**Prefer a file to Secrets?** Put `WANDB_API_KEY=your-key` in a text file on your Drive and pass its path to the setup call: `cu.project.setup_colab(URL, env_source="/content/drive/MyDrive/cs4337.env")`. This is also how the Colab CLI (SSH) session gets its keys: see [Training with the Colab CLI](#Training-with-the-Colab-CLI).


In [ ]:
cu.env.load_secrets("WANDB_API_KEY", persist_to=REPO / ".env")

## 2. Run the project (choose one option)

Both options run exactly the same training and save results to the same folder (Google Drive in Colab, `runs/` locally). Pass a config name from the project's `configs/` folder. To try your own settings, copy `config01.json` in the file browser (left sidebar, `CodeCS4337Fall2026/LitWBTrainingBasicConvnet/configs/`), e.g. to `config02.json`, edit it, and pass its name.

**Run the next cell first** (in Colab and locally): it finds the repo folder and makes the project importable.

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from LitWBTrainingBasicConvnet import main` | Simple, everything stays in the notebook |

In [ ]:
import colabutilscs4337 as cu

# Colab: REPO was set by the setup cell above. Locally the setup cells are skipped, and
# setup_colab only returns the repo folder (the parent of this project folder) and
# puts it on the import path.
if "REPO" not in globals():
    REPO = cu.project.setup_colab(
        "https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git"
    )
print(f"Repo folder: {REPO}")

### Where your results go

Run the cell below and look at what it prints: `OUTPUT_DIR` is the folder where **every run of this project is saved**: checkpoints, metrics, plots, and `runs_summary.csv`. It is set in `.env` at the repo root and read by `PROJECT`, the project's `colabutilscs4337.project.Project`.

- **In Colab** it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets. Open *MyDrive → CodeCS4337Fall2026 → runs* in Google Drive to find them later.
- **Locally** it is `runs`, i.e. the `runs/` folder of the repo (relative paths are relative to the repo root).

Each run gets its own folder, `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/<config>/<timestamp>/`.

In [ ]:
from LitWBTrainingBasicConvnet.utils import PROJECT

print(f"OUTPUT_DIR = {PROJECT.output_dir()}")
print(f"Runs of this project are saved in: {PROJECT.runs_dir()}")

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder found above. Run the cell below. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.

In [ ]:
!cd "{REPO}" && python -m LitWBTrainingBasicConvnet --config config01.json

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m LitWBTrainingBasicConvnet --config config01.json
```

or, from inside the project folder:

```bash
cd /content/CodeCS4337Fall2026/LitWBTrainingBasicConvnet
python main.py --config config01.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them). While training runs in the terminal, the notebook stays free for other work.

### Option B: Run from Python

The project can be imported because the cell at the start of this section added the repo folder to Python's import path. This works in Colab and locally.

In [ ]:
from LitWBTrainingBasicConvnet import main

main("config01.json")

## 3. Results

Each run is saved under `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/<config>/<timestamp>/`: in Colab that is Google Drive, `MyDrive/CodeCS4337Fall2026/runs/`; locally, the `runs/` folder of the repo (both set by `OUTPUT_DIR` in `.env`). `runs_summary.csv` has one row per run of a config.

**Early stopping.** `config01.json` has `"early_stopping": true` and `"patience": 3` in its `"training"` section: training stops when `val_acc` has not improved for 3 epochs in a row, so `"epochs"` is the **most** epochs a run can take. When a run stops early, it prints after which epoch; the number of epochs it actually ran is the `epochs_run` column of `runs_summary.csv` and in the W&B run's summary. The model tested is still the best checkpoint, not the last epoch. Set `"early_stopping": false` to always train all `"epochs"`.

In [ ]:
from LitWBTrainingBasicConvnet.utils import PROJECT

print((PROJECT.runs_dir() / "config01/runs_summary.csv").read_text())

## 4. Load a trained model

Running a config again trains a **new** model from scratch, unless you resume an earlier run (section 5). To use a model you already trained (after a runtime reset, to test it again, or to try it on images), load it with `load_model`: it finds the checkpoint under `<OUTPUT_DIR>/LitWBTrainingBasicConvnet/`, rebuilds the network with that run's settings, and loads its weights. Nothing is trained.

**Before this section:** you need at least one run of the config, trained in section 2 (in Colab: trained with Drive mounted, so it was saved to Drive). Then, in Colab, run the setup cells of section 1 (Drive must be mounted, that is where your runs are); in Colab and locally, run the first cell of section 2, which finds the repo folder.

| Call | What it loads |
| --- | --- |
| `load_model("config01")` | The newest run of `config01`, its best checkpoint (`best_*.ckpt`) |
| `load_model("config01/2026-10-02_12-29-09")` | One specific run: the folder name of the run, under `LitWBTrainingBasicConvnet/config01/` |
| `load_model("config01", which="last")` | The last epoch instead of the best one |
| `load_model("config01/2026-10-02_12-29-09/best_epoch07_valacc0.9260.ckpt")` | One specific checkpoint file of a run |
| `load_model("config02")` | The newest run of your own `config02.json` |

Each run folder keeps **two checkpoints**: the best epoch (`best_*.ckpt`) and the last epoch (`last.ckpt`); older ones are replaced during training. The model comes back ready for predictions (in eval mode, on the CPU; add `device="cuda"` for the GPU). It prints which checkpoint it loaded.

In [ ]:
from LitWBTrainingBasicConvnet import load_model

model = load_model("config01")

Try the loaded model on the test set: the cell below prints its predictions for the first 8 test images and its accuracy on all 10,000. Images get the same normalization as in training. The accuracy should match the run's `test_acc` in `runs_summary.csv` when you loaded the best checkpoint.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from LitWBTrainingBasicConvnet.dataloaders.fashion_mnist import (
    FASHION_MNIST_MEAN,
    FASHION_MNIST_STD,
)
from LitWBTrainingBasicConvnet.utils import PROJECT

transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(FASHION_MNIST_MEAN, FASHION_MNIST_STD)]
)
test_set = datasets.FashionMNIST(
    PROJECT.data_dir(), train=False, download=True, transform=transform
)

images, labels = zip(*(test_set[i] for i in range(8)))
with torch.no_grad():
    preds = model(torch.stack(images)).argmax(dim=1)
for pred, label in zip(preds, labels):
    print(f"predicted {test_set.classes[pred]:<12} true {test_set.classes[label]}")

with torch.no_grad():
    correct = sum(
        (model(x).argmax(dim=1) == y).sum().item()
        for x, y in DataLoader(test_set, 1000)
    )
print(f"Test accuracy: {correct / len(test_set):.4f}")

## 5. Keep training a run

`load_model` gives you a trained model to use; it does not train it further. To **continue training** an earlier run, for more epochs or to finish a run cut off by a Colab disconnect, pass `resume_from` to `main`. Training continues from the run's last checkpoint (`last.ckpt`, saved after every epoch) and runs the config's `"epochs"` **more** epochs: a 10-epoch run resumed with `config01.json` ends after epoch 20, or earlier if early stopping stops it.

**Before this section:** the same as for section 4 (a saved run; in Colab, section 1 with Drive mounted; then the first cell of section 2).

| Call | What it continues |
| --- | --- |
| `main("config01.json", resume_from="config01")` | The newest run of `config01`, for `"epochs"` more epochs. Run it again to continue further: the newest run is then the continued one. |
| `main("config01.json", resume_from="config01/2026-10-02_12-29-09")` | One specific run (its folder name) |
| `main("config01.json", resume_from="config01/2026-10-02_12-29-09/last.ckpt")` | One specific checkpoint file |
| `main("config01_more.json", resume_from="config01")` | A different number of extra epochs: a copy of the config with another `"epochs"` (saved under `config01_more/`) |

What comes from where:

- **From the checkpoint:** the weights, the epoch count, and the optimizer state, including its **learning rate** and weight decay. Changing `lr` or `weight_decay` in the config has no effect when resuming.
- **From the config:** the number of extra `"epochs"`, the data settings (e.g. `batch_size`), the seed, and logging. The **model settings must match the run's** (e.g. `dropout`): the saved weights only fit the same network.
- **Early stopping** starts counting again: a run that stopped early can be resumed, and gets the config's `"patience"` epochs to beat the **earlier run's best** `val_acc`.
- The continued run is saved as a **new run folder**, with `resumed_from` in its `hparams.json`, and gets its own row in `runs_summary.csv`. Its best checkpoint and test score are those of the continued run. The earlier run is not changed.

The same from a terminal: `python -m LitWBTrainingBasicConvnet --config config01.json --resume-from config01`.

In [ ]:
from LitWBTrainingBasicConvnet import main

# The config's "epochs" more epochs, on top of the newest config01 run.
main("config01.json", resume_from="config01")

## Training with the Colab CLI

Another way to train this project, once everything is final. Its only purpose
is the training run itself:

1. **Work locally** on the project: change the code or a config, run short
   tests, fix bugs.
2. **Use Google Colab** (this notebook) for debugging and interactive work.
3. **Use the Colab CLI** when everything is final and the only thing left is
   training. The CLI creates a Colab runtime from your local terminal and opens
   a shell on it, so there is no notebook or browser tab to keep open. On the
   runtime you set up the repo with `colabutilscs4337` and run `python -m LitWBTrainingBasicConvnet --config config01.json`.

The Colab CLI works on Linux and macOS (not Windows). For more detail, see
[google-colab-cli.md](https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/google-colab-cli.md) at the repo root or the
[official Colab CLI documentation](https://github.com/googlecolab/google-colab-cli).

### Step 1: Install the CLI

On your own machine:

```bash
uv tool install google-colab-cli     # or: pip install google-colab-cli
colab version                         # check that it works
```

The first command that talks to Colab prints a sign-in link: open it, sign in
with your Google account, and paste the code it shows back into the terminal.

### Step 2: The basic workflow (sessions)

A **session** is one Colab runtime (a virtual machine) with a name you choose,
here `cs4337`. These commands run in your **local** terminal:

| What | Command |
| --- | --- |
| Create a session with a GPU | `colab new -s cs4337 --gpu T4` (or `L4`, `A100`, `H100`, depending on your plan) |
| Create a session without a GPU | `colab new -s cs4337` (CPU only) |
| List your sessions / check one | `colab sessions` / `colab status -s cs4337` |
| Attach to a session (open a shell on it) | `colab ssh -s cs4337` |
| Leave the shell; the session keeps running | `exit` |
| Stop the session (delete the runtime) | `colab stop -s cs4337` |

Use a GPU runtime: `--gpu T4` is enough for this project. A CPU session can't be switched to a GPU: stop it and create a new one.

> **Once a session is stopped, nothing on it stays.** The cloned repo, the
> downloaded data, and every file that is not on Google Drive are deleted.
> This project writes its runs to Google Drive (step 4), so mount Drive before
> training.

### Step 3: Mount Google Drive

Drive is mounted from your local terminal, not from inside the session. If you
are attached, `exit` first:

```bash
exit                                # only if you are inside the session
colab drivemount -s cs4337          # open the link it prints and allow access
colab ssh -s cs4337                 # attach again
ls /content/drive/MyDrive           # check: your Drive files are listed
```

Nothing in the project mounts Drive again: the setup in step 4 uses this mount.

### Step 4: Set up the repo and train

One call of the course library
[`colabutilscs4337`](https://txstcodeplayground.github.io/ColabUtilsCS4337/guide/course_projects.html) sets up everything the
runtime needs: it clones the repo (or pulls it, if it is already there), writes
`.env`, and installs only the packages Colab doesn't already have (a plain
`pip install -r requirements.txt` would replace Colab's own versions, which is
slow and can break other preinstalled packages). The session already has Drive
mounted (step 3), so nothing is mounted again. Inside the session:

```bash
pip install -q colabutilscs4337
python - <<'EOF'
import colabutilscs4337 as cu

cu.project.setup_colab(
    "https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git",
    env_source="/content/drive/MyDrive/cs4337.env",  # your env file on Google Drive
    mount=False,  # `colab drivemount` already mounted Drive
)
EOF
cd /content/CodeCS4337Fall2026
```

The runtime only sees what is on GitHub: if you changed code or a config
locally, push it to your own fork first and use that URL instead.

**You must provide your env file on Google Drive.** Colab Secrets can only be
read from a notebook, not from an SSH session, so the keys come from a plain
text file on your Drive: one `NAME=value` per line, e.g. `WANDB_API_KEY=your-key`.
`env_source` is its path, absolute (`/content/drive/MyDrive/...`) or relative
to `MyDrive`. If the file is not there, `setup_colab` stops with an error that
names the path it looked for. The resulting `.env` is `.envcolab` (which sets
`DATA_DIR=/content/data`, data on the runtime's fast local disk, and
`OUTPUT_DIR=/content/drive/MyDrive/CodeCS4337Fall2026/runs`, runs on Drive so they
are kept after a stop) with your file added below it.

Create the file once, from the session. `read -s` hides the key while you
paste it, so it is not shown on screen or saved in the shell history. Get the
W&B API key from [wandb.ai/authorize](https://wandb.ai/authorize):

```bash
read -rsp "W&B API key: " key && echo "WANDB_API_KEY=$key" >> /content/drive/MyDrive/cs4337.env && unset key && echo
```

Without a key the run still trains, but W&B logs offline into the run folder;
upload it later with the `wandb sync` command that the run prints. Never put the key in `.envcolab`, which is in git.

Start a `tmux` session first, so training keeps going if the SSH connection
drops or you close your laptop, then run the training from the repo root:

```bash
tmux new -s train                  # if tmux is missing: apt-get install -y tmux
python -m LitWBTrainingBasicConvnet --config config01.json
```

To leave training running, press `Ctrl+B`, then `D`. To come back later:
`colab ssh -s cs4337`, then `tmux attach -t train`.

### Step 5: Check that everything was logged

The run folders are on Google Drive, so they are kept after the session
stops. From the session:

```bash
ls /content/drive/MyDrive/CodeCS4337Fall2026/runs/LitWBTrainingBasicConvnet/config01
```

or on [drive.google.com](https://drive.google.com), under **My Drive >
CodeCS4337Fall2026 > runs > LitWBTrainingBasicConvnet**. Check that:

- each run has its own `<timestamp>` folder (the path is printed when the run starts), with its config, results, plots, and checkpoints;
- `runs_summary.csv`, next to the run folders, has one new row per run;
- the run shows up in your W&B project: the run prints its W&B link. If it printed a `wandb sync` command instead, the run was logged offline (no key or no login); run that command later, from a machine with your key, to upload it;
- if the session stopped or the run was cut off, start a new session, repeat the steps above, and continue from the last checkpoint on Drive with `--resume-from config01`, e.g. `python -m LitWBTrainingBasicConvnet --config config01.json --resume-from config01`.

When everything is checked, stop the session so it no longer uses your Colab
quota:

```bash
colab stop -s cs4337
```

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two checkpoints are saved to Drive *during* training: the best model so far (`best_epochXX_valaccY.ckpt`), every time validation accuracy improves, and the last epoch (`last.ckpt`), after every epoch. If the runtime disconnects mid-run, both are kept.
- The run's other files (results, plots, and the row in `runs_summary.csv`) are written only when the run finishes. To finish an interrupted run, resume it (section 5): it continues from the last saved epoch.
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 1`), then start the full run.
- Prefer several shorter runs over one very long run; each run saves its own results.
- After a disconnect, re-run the setup cells (including the API-key cell), then resume the run with `resume_from` (section 5) instead of starting over.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.